# Sports facility pricing simulation

Kuandyk Bibarys and Tulebay Aslan.

This notebook inspects the completed synthetic-data experiment. Its outputs below were executed against the saved run. The training and inference implementation is in `src/run_analysis.py`. No real aggregator data are used. The known simulator parameters are reserved for evaluation.

Run this notebook from the repository root.

In [1]:
from pathlib import Path
import json
import pandas as pd
ROOT = Path.cwd()
assert (ROOT / "src/run_analysis.py").exists(), "Open this notebook from the repository root"
summary = json.loads((ROOT / "results/summary.json").read_text())
print("Selected model:", summary["selected"])
print("Recorded runtime (minutes):", round(summary["runtime_seconds"] / 60, 2))
print("H1, H2, H3:", summary["h1"], summary["h2"], summary["h3"])

Selected model: LightGBM
Recorded runtime (minutes): 6.03
H1, H2, H3: False True False


## Data audit

The original CSV was unchanged. Generator seed 2026 reproduced its bytes exactly. The training and test periods are chronological.

In [2]:
print(json.dumps(json.loads((ROOT / "results/dataset_audit.json").read_text()), indent=2))
print(pd.read_csv(ROOT / "results/tables/validation_folds.csv").to_string(index=False))

{
  "rows_before": 153300,
  "duplicate_slots": 0,
  "invalid_prices": 0,
  "missing_cells": 0,
  "rows_after": 153300,
  "venues": 14,
  "start": "2024-09-01 08:00:00",
  "end": "2026-08-31 22:00:00",
  "train_n": 122640,
  "test_n": 30660,
  "train_booked": 31888,
  "test_booked": 9988,
  "train_rate": 0.2600130463144162,
  "test_rate": 0.32576647097195044,
  "test_start": "2026-04-08 08:00:00",
  "source_sha256": "fc17c31a84abf57ab320e560b5244cdf7574cdf30e0e12c9c54343ec70b8f05a"
}
 fold  train_n  validation_n           train_end    validation_start      validation_end
    1    23100         24514 2024-12-19 22:00:00 2024-12-27 08:00:00 2025-04-22 18:00:00
    2    47614         24472 2025-04-15 18:00:00 2025-04-22 19:00:00 2025-08-17 11:00:00
    3    72086         24514 2025-08-10 11:00:00 2025-08-17 12:00:00 2025-12-11 22:00:00
    4    96600         24570 2025-12-04 22:00:00 2025-12-12 08:00:00 2026-04-07 22:00:00


## Forecasting and uncertainty

The table uses averaged predictions for the five-seed nonlinear models. Per-seed scores are retained separately. Comparisons against the LSTM do not establish a significant LightGBM advantage.

In [3]:
for name in ["cv_results", "forecast_metrics", "per_seed_metrics", "paired_tests"]:
    print("\n" + name)
    print(pd.read_csv(ROOT / ("results/tables/" + name + ".csv")).to_string(index=False))


cv_results
   model  mean_logloss  fold_sd                                                                        fold_scores
Logistic      0.532918 0.072851 [0.47674890343422605, 0.6284301154474887, 0.5511283115875334, 0.47536329447161657]
LightGBM      0.503082 0.062861    [0.4572615015371972, 0.5807825198223878, 0.5272006109793239, 0.447082681449468]
    LSTM      0.513695 0.059001   [0.48332404253387834, 0.5855995201183457, 0.534691140230911, 0.4511670650757385]

forecast_metrics
      model    brier  logloss      auc      ece      bss    slope  intercept  brier_lo  brier_hi  logloss_lo  logloss_hi  mae_bookings_day  wape_percent
Climatology 0.210672 0.618082 0.645267 0.066540 0.000000 0.691193   0.038844  0.205810  0.215542    0.606170    0.630061          1.809798     37.036725
   Logistic 0.209182 0.607211 0.635509 0.030098 0.007071 0.748565  -0.162757  0.205325  0.212846    0.598341    0.615607          1.677211     34.323390
   LightGBM 0.195080 0.574702 0.697329 0.004328 0.0

## Counterfactual policy and known simulator response

The estimated revenue effect must be read together with the simulator check. Both evaluate exactly the same chosen prices. Neither is a real commercial trial.

In [4]:
for name in ["policy_effects", "policy_truth", "weather_ablation", "group_errors"]:
    print("\n" + name)
    print(pd.read_csv(ROOT / ("results/tables/" + name + ".csv")).to_string(index=False))


policy_effects
         metric  estimate        lo        hi
revenue_percent  4.874898  3.278939  6.792339
 utilization_pp  0.562297  0.022473  1.163555
     offpeak_pp -0.310300 -0.935465  0.332883
cancellation_pp -0.511748 -0.686928 -0.323047

policy_truth
         metric     truth
revenue_percent  1.146234
 utilization_pp -0.556896
     offpeak_pp -0.923484
cancellation_pp -0.941780

weather_ablation
 metric  without_weather  with_weather  difference       lo       hi
  brier         0.197370      0.195080    0.002290 0.001713 0.002859
logloss         0.580558      0.574702    0.005856 0.004429 0.007251

group_errors
  grouping    group     n  booked    brier  logloss  mean_pred  observed
venue_type football  8760    3000 0.193255 0.568763   0.329729  0.342466
venue_type   futsal  8760    2693 0.194320 0.574877   0.311215  0.307420
venue_type swimming  6570    2357 0.209066 0.607387   0.363708  0.358752
venue_type   tennis  6570    1938 0.184541 0.549703   0.289916  0.294977
    se

## Reproduce the full run

Set `RUN_FULL_PIPELINE = True` before executing the next cell to retrain every model and overwrite the derived outputs. This cell is intentionally disabled in the saved notebook to avoid triggering an expensive rerun just by opening it.

In [5]:
import subprocess, sys
RUN_FULL_PIPELINE = False
if RUN_FULL_PIPELINE:
    subprocess.run([sys.executable, "src/run_analysis.py"], check=True)
    subprocess.run([sys.executable, "src/check_artifacts.py"], check=True)
    subprocess.run([sys.executable, "src/build_results_section.py"], check=True)
else:
    print("Full rerun disabled. The saved outputs above come from the completed analysis.")

Full rerun disabled. The saved outputs above come from the completed analysis.
